In [10]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

In [2]:
llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")

In [ ]:
def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

def add(a: int, b: int) -> int:
    """Adds a and b.

    Args:
        a: first int
        b: second int
    """
    return a + b

In [ ]:
tools = [add, multiply]
llm_with_tools = llm.bind_tools(tools)

sys_msg = SystemMessage(
    content="Use either tools to perform arithmetic on a set of inputs or just give me answer with your knowledge")


In [ ]:
def tool_calling_llm(state: MessagesState):
   return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

In [ ]:
# Graph
builder = StateGraph(MessagesState)

# Define nodes: these do the work
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

# Define edges: these determine how the control flow moves
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
    # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
    tools_condition,
)
builder.add_edge("tools", "tool_calling_llm")

In [7]:
react_graph = builder.compile()

In [ ]:
human_message = HumanMessage(content="Add 3 and 4")
response = react_graph.invoke({"messages": [human_messages]})
for m in response['messages']:
    m.pretty_print()

================================ Human Message =================================

Add 3 and 4
================================== Ai Message ==================================
Tool Calls:
  add (711943bd-4824-4f86-b494-4e8561144cd4)
 Call ID: 711943bd-4824-4f86-b494-4e8561144cd4
  Args:
    a: 3
    b: 4
================================= Tool Message =================================
Name: add

7
================================== Ai Message ==================================

The sum of 3 and 4 is 7.


In [ ]:
human_message = HumanMessage(content="Multiply that by 10")
response = react_graph.invoke({"messages": [human_message]})
for m in response['messages']:
    m.pretty_print()

================================ Human Message =================================

Multiply that by 10
================================== Ai Message ==================================

I need a number to multiply by 10. What number would you like to use?
